# ✈️ Airline Satisfaction v2 — Routes, Original-Data Lookups & a Dual-T4 GPU Stack
### Playground Series S6E10 · XGBoost ×2 · CatBoost · RealMLP · Logistic stacking

**Metric:** ROC-AUC on the probability of `satisfaction`.

v1 of this notebook plateaued at ~0.9590 CV / 0.9582 LB. The gap to 0.96+ is not tuning — it is **features**.
This version adds the feature blocks that public work in this competition measured, one at a time, on a single
LightGBM (gains below are those published measurements, not re-measured here):

| Block | Idea | Published gain |
|---|---|---|
| `org` | For every column, P(satisfied \| value) **looked up in the original dataset** | +0.00104 |
| `origm` | Prediction of a model trained **only on the original dataset** | +0.00029 |
| `digit` | Individual digits of the numeric columns (synthetic-generator artefacts) | +0.00021 |
| `freq` | How common each value is | +0.00013 |
| `te` / `te2` | In-fold target encoding of every column, bins and key pairs | +0.00022 |
| `route` | `Flight Distance` is also a **route ID**: route-level profile + each passenger's deviation from it | +0.00013 |

Then four diverse models on identical folds, stacked by a logistic regression on their logits.

**Inputs to attach (Add Input):**
1. the competition data — `playground-series-s6e10`;
2. the original dataset — **`teejmahal20/airline-passenger-satisfaction`** (search "Airline Passenger Satisfaction").

**Settings:** Accelerator **GPU T4 ×2**, **Internet ON** (RealMLP's library `pytabkit` is pip-installed).
Expected runtime ≈ 1–1.5 h. Every GPU stage uses both T4s.

## 0. Setup
Imports and switches. Off Kaggle the notebook runs a **DEBUG** mode (small sample, 3 folds, tiny models) so the
whole pipeline can be smoke-tested on a CPU before spending GPU quota.

In [ ]:
import os, sys, glob, time, queue, subprocess, contextlib, warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import torch
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from scipy.special import logit
from scipy.stats import rankdata

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

ON_KAGGLE = os.path.exists('/kaggle/input')
DEBUG = not ON_KAGGLE
SEED = 42
TARGET = 'satisfaction'
N_FOLDS = 3 if DEBUG else 5

print('xgboost', xgb.__version__, '| catboost', cb.__version__, '| lightgbm', lgb.__version__, '| torch', torch.__version__)
print('ON_KAGGLE =', ON_KAGGLE, '| DEBUG =', DEBUG)

### GPU discovery and the two-GPU pool
A thread-safe queue holds the GPU ids. A job borrows one, trains on it and gives it back, so a 2-worker pool keeps
**GPU 0 and GPU 1 busy at the same time** (XGBoost releases the GIL while training). CatBoost instead spans both
GPUs inside one model, and RealMLP runs as one separate process per GPU (section 7).

In [ ]:
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}  {p.total_memory / 1e9:.1f} GB')
if N_GPUS == 0:
    print('No GPU visible -> CPU mode (fine for DEBUG; on Kaggle select Accelerator = GPU T4 x2)')
if ON_KAGGLE and N_GPUS < 2:
    print('WARNING: fewer than 2 GPUs visible - select "GPU T4 x2" for full speed.')

N_WORKERS = max(N_GPUS, 1)
GPU_POOL = queue.Queue()
for g in range(N_WORKERS):
    GPU_POOL.put(g)

@contextlib.contextmanager
def gpu_slot():
    g = GPU_POOL.get()
    try:
        yield g
    finally:
        GPU_POOL.put(g)

## 1. Load the competition data and the original dataset
Paths are discovered with globs, so the notebook does not depend on Kaggle's exact mount folders. The original
dataset is recognised by its columns, so **any Kaggle copy** of the 129,880-row survey works (e.g.
`teejmahal20/airline-passenger-satisfaction` or `arseniyshutko/binary-aviation-satisfaction-129k`). Its label may be
text (`satisfied` / `neutral or dissatisfied`) or boolean; both are handled.

If no copy is attached, the notebook downloads `teejmahal20/airline-passenger-satisfaction` with `kagglehub`
(requires **Internet ON**). To point at a specific file instead, set `ORIGINAL_PATH`.

In [ ]:
def find_competition_dir():
    for p in glob.glob('/kaggle/input/**/train.csv', recursive=True):
        d = os.path.dirname(p)
        if os.path.exists(os.path.join(d, 'sample_submission.csv')):
            return d
    raise FileNotFoundError('Competition data not attached: Add Input -> playground-series-s6e10')

ORIGINAL_PATH = None     # optional manual override: a folder or a .csv path of the original dataset
NEEDED = {'satisfaction', 'online boarding', 'flight distance', 'class'}

def survey_csvs(root, skip_dir=None):
    """All CSVs under root whose header looks like the original survey, grouped by folder."""
    by_dir, seen = {}, []
    for d, _, files in os.walk(root, followlinks=True):
        for f in files:
            p = os.path.join(d, f); seen.append(p)
            if not f.lower().endswith('.csv') or d == skip_dir or 'playground-series' in p.lower():
                continue
            try:
                cols = {c.strip().lower() for c in pd.read_csv(p, nrows=2).columns}
            except Exception:
                continue
            if NEEDED <= cols:
                by_dir.setdefault(d, []).append(p)
    return by_dir, seen

def find_original_files():
    if ORIGINAL_PATH:
        return [ORIGINAL_PATH] if ORIGINAL_PATH.lower().endswith('.csv') else sorted(glob.glob(f'{ORIGINAL_PATH}/*.csv'))
    by_dir, seen = survey_csvs('/kaggle/input', skip_dir=DATA_DIR)
    if not by_dir:                              # not attached -> download it (needs Internet ON)
        print('Original dataset not found under /kaggle/input; trying a download with kagglehub ...')
        try:
            import kagglehub
            by_dir, _ = survey_csvs(kagglehub.dataset_download('teejmahal20/airline-passenger-satisfaction'))
        except Exception as e:
            print('  download failed:', repr(e))
    if not by_dir:
        print('Files found under /kaggle/input:', *seen[:100], sep='\n  ')
        raise FileNotFoundError('Original dataset not found. Attach it (Add Input -> "Airline Passenger Satisfaction"), '
                                'turn Internet ON, or set ORIGINAL_PATH to its folder.')
    return sorted(by_dir[sorted(by_dir)[0]])   # if several copies are attached, use just one of them

if ON_KAGGLE:
    DATA_DIR, OUT_DIR = find_competition_dir(), '/kaggle/working'
    ORIG_FILES = find_original_files()
else:
    DATA_DIR = os.environ.get('AIRLINE_DATA', '.')
    OUT_DIR = '.'
    ORIG_FILES = sorted(glob.glob(os.path.join(DATA_DIR, 'original', '*.csv')))
print('competition:', DATA_DIR, '| original:', ORIG_FILES)

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

orig = pd.concat([pd.read_csv(f) for f in ORIG_FILES], ignore_index=True)
canon = {c.lower(): c for c in train.columns}                        # align column names with the competition's
orig.columns = [canon.get(c.strip().lower(), c.strip()) for c in orig.columns]
orig = orig.drop(columns=[c for c in ('Unnamed: 0', 'id', 'Inflight service') if c in orig.columns])
orig[TARGET] = orig[TARGET].astype(str).str.strip().str.lower().isin(['satisfied', 'true', '1', '1.0']).astype(int)
assert 0.3 < orig[TARGET].mean() < 0.6, f'unexpected original label values: {orig[TARGET].unique()[:5]}'

if DEBUG:
    train = train.sample(30_000, random_state=SEED).reset_index(drop=True)
    test = test.sample(10_000, random_state=SEED).reset_index(drop=True)
    sample = sample.set_index('id').loc[test['id']].reset_index()

train[TARGET] = train[TARGET].astype(str).str.lower().eq('true').astype(int)
y = train[TARGET].values
print(f'train {train.shape} | test {test.shape} | original {orig.shape}')
print(f'satisfied rate: train {y.mean():.4f} | original {orig[TARGET].mean():.4f}')

## 2. Two facts that drive the features
**(a) `Flight Distance` behaves like a route ID.** It has ~3,500 distinct values. If it were just a length, the
satisfaction rate would change smoothly from one value to the next. Below, each dot is one distance value; the
spread of the rates around their smooth trend is far larger than sampling noise, so the exact value carries
information of its own — it names a route.

**(b) The original dataset works as a lookup, not as extra rows.** The competition data was synthesised from the
original survey. Appending the original rows *hurt* in v1, but each column's satisfaction rate **in the original**
is a strong single feature on the synthetic data (AUCs printed below).

In [ ]:
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
RATINGS = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location',
           'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service',
           'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness']
NUMS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
BASE = CATS + NUMS + RATINGS
for df in (train, test, orig):
    df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(0)

# (a) route evidence
g = train.groupby('Flight Distance')[TARGET].agg(['mean', 'size']).query('size >= 30')
smooth = g['mean'].rolling(11, center=True, min_periods=3).mean()
noise = np.sqrt((y.mean() * (1 - y.mean()) / g['size']).mean())
print(f'distance values with 30+ rows: {len(g)} | spread around smooth trend {(g["mean"] - smooth).std():.3f} '
      f'vs sampling noise {noise:.3f}')
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.scatter(g.index, g['mean'], s=5, color='#2a78d6', alpha=0.45, label='rate of one distance value')
ax.plot(g.index, smooth, color='#0b0b0b', lw=1.2, label='smooth trend (11 neighbours)')
ax.set_xlabel('Flight Distance'); ax.set_ylabel('share satisfied'); ax.legend(loc='lower right', markerscale=3)
ax.set_title('Flight Distance: a smooth length plus a route-specific key')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

# (b) original-dataset lookups as single features
look = {}
for c in BASE:
    m = train[c].map(orig.groupby(c)[TARGET].mean()).fillna(orig[TARGET].mean())
    look[c] = roc_auc_score(y, m)
display(pd.Series(look, name='AUC of P(satisfied | value) from original').sort_values(ascending=False).round(4).to_frame().T)

## 3. Label-free feature blocks (built once on train + test)
None of these use the competition labels, so they are safe to build on train and test together.

* **`org`** — for each base column and each key pair: P(satisfied | value) in the original dataset (global rate for
  unseen values).
* **`origm`** — a LightGBM trained on the original dataset only; its predicted probability for every competition row.
* **`digit`** — units / tens / hundreds / thousands digit of Age, distance and both delays.
* **`freq`** — frequency of each value (base columns and key pairs).
* **bins** — distance at 10 / 100 / 500, age at 5, departure delay at 10 (used as target-encoding keys).
* **`route`** — per distance value: row count, the mean of every other column, and each passenger's deviation
  from that mean.

In [ ]:
PAIRS = [('Class', 'Type of Travel'), ('Customer Type', 'Type of Travel'), ('Class', 'Customer Type'),
         ('Gender', 'Class'), ('Inflight wifi service', 'Online boarding'),
         ('Inflight wifi service', 'Type of Travel'), ('Online boarding', 'Type of Travel'),
         ('Seat comfort', 'Inflight entertainment'), ('Class', 'Inflight wifi service'),
         ('Class', 'Online boarding')]
ROUTE_PAIRS = [('Flight Distance', 'Class'), ('Flight Distance', 'Type of Travel'),
               ('Flight Distance', 'Customer Type'), ('Age', 'Class')]

n_train = len(train)
full = pd.concat([train[BASE], test[BASE]], ignore_index=True)
both = pd.concat([full, orig[BASE]], ignore_index=True)        # shared integer codes for competition + original

codes = {c: pd.factorize(both[c].astype(str), sort=True)[0].astype(np.int64) for c in BASE}
full['dist_10'] = full['Flight Distance'] // 10
full['dist_100'] = full['Flight Distance'] // 100
full['dist_500'] = full['Flight Distance'] // 500
full['age_5'] = full['Age'] // 5
full['dep_delay_10'] = full['Departure Delay in Minutes'] // 10
BINS = ['dist_10', 'dist_100', 'dist_500', 'age_5', 'dep_delay_10']
for b in BINS:
    codes[b] = np.r_[pd.factorize(full[b], sort=True)[0], np.full(len(orig), -1)].astype(np.int64)

def combo_key(cols, n=None):
    """One compact integer id per combination of the given columns (over competition + original rows)."""
    key = np.zeros(len(both), dtype=np.int64)
    for c in cols:
        key = key * (codes[c].max() + 2) + (codes[c] + 1)
    return pd.factorize(key)[0]

F = pd.DataFrame(index=full.index)
for c in BASE:                                     # raw columns (categoricals as category dtype)
    F[c] = full[c].astype(str).astype('category') if c in CATS else full[c].astype('float32')

# org: original-dataset lookups (singles + pairs)
o_y = orig[TARGET].values.astype(float); o_rate = o_y.mean(); n_full = len(full)
for grp in [(c,) for c in BASE] + PAIRS + ROUTE_PAIRS:
    k = combo_key(grp); nk = k.max() + 1
    s = np.bincount(k[n_full:], weights=o_y, minlength=nk); cnt = np.bincount(k[n_full:], minlength=nk)
    rate = np.where(cnt > 0, s / np.maximum(cnt, 1), o_rate)
    F['org__' + '__'.join(grp)] = rate[k[:n_full]].astype('float32')

# origm: model trained on the original only
Xo = orig[BASE].copy(); Xf = full[BASE].copy()
for c in CATS:
    dt = pd.CategoricalDtype(sorted(set(Xo[c].astype(str)) | set(Xf[c].astype(str))))
    Xo[c] = Xo[c].astype(str).astype(dt); Xf[c] = Xf[c].astype(str).astype(dt)
om = lgb.LGBMClassifier(n_estimators=100 if DEBUG else 600, learning_rate=0.05, num_leaves=63, colsample_bytree=0.6,
                        subsample=0.8, subsample_freq=1, reg_lambda=5.0, random_state=SEED, verbose=-1)
om.fit(Xo, orig[TARGET])
F['orig_model'] = om.predict_proba(Xf)[:, 1].astype('float32')
print(f'original-only model, AUC on competition train: {roc_auc_score(y, F["orig_model"].values[:n_train]):.5f}')

# digit features
for c in NUMS:
    v = full[c].fillna(0).astype(np.int64)
    for k in range(4):
        d = (v // 10 ** k % 10).astype('int8')
        if d.nunique() > 1:
            F[f'{c}_digit{k}'] = d

# frequency of values and of key pairs
for grp in [(c,) for c in BASE] + PAIRS:
    k = combo_key(grp)[:n_full]
    F['freq__' + '__'.join(grp)] = (np.bincount(k)[k] / n_full).astype('float32')

for b in BINS:
    F[b] = full[b].astype('float32')

# route profile
coded = full[BASE].copy()
for c in CATS:
    coded[c] = coded[c].astype('category').cat.codes
grp = coded.groupby(full['Flight Distance'])
F['fd_count'] = full['Flight Distance'].map(full['Flight Distance'].value_counts()).astype('float32')
for c in BASE:
    if c == 'Flight Distance':
        continue
    m = grp[c].transform('mean')
    F[f'fdm__{c}'] = m.astype('float32')
    F[f'fdd__{c}'] = (coded[c] - m).astype('float32')

X_all = F
print(X_all.shape[1], 'label-free features')

## 4. In-fold target encodings (`te`, `te2`)
Smoothed satisfaction rate of every base column, every bin and every key/route pair — computed from the
**competition** labels, so it must be leak-free:
* validation and test rows are encoded with a mapping fitted on the fold's training part;
* training rows are **cross-fitted** (inner 5-fold), so no row is ever encoded with its own label.

Integer keys + `np.bincount` make this fast enough to redo inside every fold.

In [ ]:
TE_GROUPS = [(c,) for c in BASE + BINS] + PAIRS + ROUTE_PAIRS
TE_KEYS = np.stack([combo_key(gr)[:len(full)] for gr in TE_GROUPS], axis=1).astype(np.int32)
TE_KEYS = np.stack([pd.factorize(TE_KEYS[:, j])[0] for j in range(TE_KEYS.shape[1])], axis=1).astype(np.int32)
TE_NK = TE_KEYS.max(axis=0) + 1
TE_NAMES = ['te__' + '__'.join(gr) for gr in TE_GROUPS]
KEYS_TR, KEYS_TE = TE_KEYS[:n_train], TE_KEYS[n_train:]
TE_SMOOTH = 20.0
print(len(TE_GROUPS), 'target-encoded keys')

def _enc(keys, yy, nk):
    prior = yy.mean()
    s = np.bincount(keys, weights=yy, minlength=nk); c = np.bincount(keys, minlength=nk)
    return ((s + TE_SMOOTH * prior) / (c + TE_SMOOTH)).astype(np.float32)

def te_frames(tr, va, seed=SEED):
    ytr = y[tr].astype(np.float64)
    inner = list(KFold(5, shuffle=True, random_state=seed).split(tr))
    A = np.empty((len(tr), len(TE_GROUPS)), np.float32)
    V = np.empty((len(va), len(TE_GROUPS)), np.float32)
    T = np.empty((len(KEYS_TE), len(TE_GROUPS)), np.float32)
    for j, nk in enumerate(TE_NK):
        k = KEYS_TR[tr, j]
        mp = _enc(k, ytr, nk)
        V[:, j] = mp[KEYS_TR[va, j]]; T[:, j] = mp[KEYS_TE[:, j]]
        for a, b in inner:
            A[b, j] = _enc(k[a], ytr[a], nk)[k[b]]
    mk = lambda M: pd.DataFrame(M, columns=TE_NAMES)
    return mk(A), mk(V), mk(T)

X = X_all.iloc[:n_train].reset_index(drop=True)
X_test = X_all.iloc[n_train:].reset_index(drop=True)

def fold_data(tr, va):
    """Full feature frames for one fold: label-free blocks + in-fold target encodings."""
    a, v, t = te_frames(tr, va)
    return (pd.concat([X.iloc[tr].reset_index(drop=True), a], axis=1),
            pd.concat([X.iloc[va].reset_index(drop=True), v], axis=1),
            pd.concat([X_test, t], axis=1))

FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, y))
print('total features per model:', X.shape[1] + len(TE_NAMES))

## 5. Cross-validation engine
All models share **the same folds**, which keeps the stacking in section 8 clean. Each job borrows a GPU from the
pool; with two workers, two folds train simultaneously. OOF and test predictions are saved as `.npy` files.

In [ ]:
def run_cv(name, fit_fn, parallel=True):
    oof = np.zeros(n_train); pred = np.zeros(len(X_test))

    def work(job):
        k, (tr, va) = job
        with gpu_slot() as g:
            t0 = time.time()
            pv, pt = fit_fn(tr, va, g, k)
        print(f'  [{name}] fold {k} on GPU {g}: AUC {roc_auc_score(y[va], pv):.5f}  ({time.time() - t0:.0f}s)', flush=True)
        return va, pv, pt

    t0 = time.time()
    with ThreadPoolExecutor(N_WORKERS if parallel else 1) as ex:
        for va, pv, pt in ex.map(work, enumerate(FOLDS)):
            oof[va] = pv; pred += pt / N_FOLDS
    score = roc_auc_score(y, oof)
    print(f'>>> {name}: OOF AUC {score:.5f}   ({(time.time() - t0) / 60:.1f} min)')
    np.save(f'{OUT_DIR}/oof_{name}.npy', oof); np.save(f'{OUT_DIR}/test_{name}.npy', pred)
    return dict(oof=oof, pred=pred, score=score)

results = {}

## 6. Gradient-boosted trees on GPU
### 6.1 XGBoost — deep and shallow variants
Two deliberately different XGBoost models (deep trees / low learning rate vs. shallow trees / heavy column
sampling) so the stack gets two views of the same features. Folds run two-at-a-time, one per GPU.

In [ ]:
XGB_VARIANTS = {
    'xgb_deep': dict(max_depth=8, learning_rate=0.02, subsample=0.9, colsample_bytree=0.5, min_child_weight=5,
                     reg_lambda=5.0, reg_alpha=0.05),
    'xgb_shallow': dict(max_depth=5, learning_rate=0.03, subsample=0.8, colsample_bytree=0.3, min_child_weight=20,
                        reg_lambda=1.0, reg_alpha=0.0),
}

def make_xgb_fit(params):
    def fit(tr, va, g, k):
        Xa, Xv, Xt = fold_data(tr, va)
        p = dict(objective='binary:logistic', eval_metric='auc', tree_method='hist', max_cat_to_onehot=8,
                 max_bin=256, device=f'cuda:{g}' if N_GPUS else 'cpu', seed=SEED + k, verbosity=0, **params)
        dtr = xgb.DMatrix(Xa, y[tr], enable_categorical=True)
        dva = xgb.DMatrix(Xv, y[va], enable_categorical=True)
        m = xgb.train(p, dtr, 200 if DEBUG else 20000, evals=[(dva, 'va')], early_stopping_rounds=300, verbose_eval=False)
        rng = (0, m.best_iteration + 1)
        return m.predict(dva, iteration_range=rng), m.predict(xgb.DMatrix(Xt, enable_categorical=True), iteration_range=rng)
    return fit

for name, params in XGB_VARIANTS.items():
    results[name] = run_cv(name, make_xgb_fit(params))

### 6.2 CatBoost across both GPUs
`devices='0:1'` trains each CatBoost model on **both T4s**, so folds run one after another. The four true
categoricals are declared as categorical (CatBoost builds its own ordered target statistics for them).
Early stopping uses `Logloss`, which is computed on the GPU; `AUC` would be computed on the CPU and slow training.

In [ ]:
CB_PARAMS = dict(iterations=200 if DEBUG else 12000, learning_rate=0.05, depth=8, l2_leaf_reg=5.0, border_count=254,
                 loss_function='Logloss', eval_metric='Logloss', od_type='Iter', od_wait=400,
                 use_best_model=True, verbose=0)
CB_PARAMS.update(dict(task_type='GPU', devices=':'.join(map(str, range(N_GPUS)))) if N_GPUS
                 else dict(task_type='CPU', thread_count=-1))

def fit_cb(tr, va, g, k):
    Xa, Xv, Xt = fold_data(tr, va)
    for D in (Xa, Xv, Xt):
        for c in CATS:
            D[c] = D[c].astype(str)
    m = cb.CatBoostClassifier(**CB_PARAMS, random_seed=SEED + k)
    m.fit(cb.Pool(Xa, y[tr], cat_features=CATS), eval_set=cb.Pool(Xv, y[va], cat_features=CATS))
    return m.predict_proba(Xv)[:, 1], m.predict_proba(cb.Pool(Xt, cat_features=CATS))[:, 1]

results['catboost'] = run_cv('catboost', fit_cb, parallel=False)

## 7. RealMLP (pytabkit) — one worker process per GPU
A well-configured neural net is as accurate as the trees here **and wrong in different places**, which is what makes
a stack gain. We use RealMLP from `pytabkit` with the configuration and feature engineering published by
Vladimir Demidov (every numeric column also as a category, count encodings, three interaction keys
target-encoded inside each fold).

**Using both GPUs:** PyTorch-Lightning (used by pytabkit) is happiest in its own process, so the folds are split
in two shards and each shard runs in a separate Python process pinned to one GPU via `CUDA_VISIBLE_DEVICES`.
The worker script is written to disk below; the next cell launches it.

In [ ]:
try:
    import pytabkit
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytabkit'], check=True)
    import pytabkit
from importlib.metadata import version
print('pytabkit', version('pytabkit'))

In [ ]:
%%writefile realmlp_worker.py
# Trains RealMLP on a subset of folds. Usage: python realmlp_worker.py <work_dir> <fold,fold,...> <device>
import sys, time, logging, warnings
import numpy as np, pandas as pd, torch
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score
from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings('ignore')
for n in ('lightning', 'lightning.pytorch', 'lightning.fabric', 'pytorch_lightning', 'lightning_fabric'):
    logging.getLogger(n).setLevel(logging.ERROR)

work_dir, folds, device = sys.argv[1], [int(f) for f in sys.argv[2].split(',')], sys.argv[3]
d = pd.read_pickle(f'{work_dir}/rmlp_data.pkl')
X, X_test, y, FOLDS = d['X'], d['X_test'], d['y'], d['folds']
for k in folds:
    t0 = time.time()
    torch.manual_seed(d['seed'] + k)
    tr, va = FOLDS[k]
    Xa, Xv, Xt = X.iloc[tr].copy(), X.iloc[va].copy(), X_test.copy()
    te = TargetEncoder(cv=5, smooth='auto', shuffle=True, random_state=d['seed'])
    names = [f'_{c}TE' for c in d['combo_names']]
    Xa[names] = te.fit_transform(Xa[d['combo_names']], y[tr])
    Xv[names] = te.transform(Xv[d['combo_names']]); Xt[names] = te.transform(Xt[d['combo_names']])
    Xa, Xv, Xt = (D.drop(columns=d['drop_after_te']) for D in (Xa, Xv, Xt))
    m = RealMLP_TD_Classifier(**d['params'], device=device)
    m.fit(Xa, y[tr], Xv, y[va])
    pv, pt = m.predict_proba(Xv)[:, 1], m.predict_proba(Xt)[:, 1]
    np.save(f'{work_dir}/rmlp_oof_{k}.npy', pv); np.save(f'{work_dir}/rmlp_test_{k}.npy', pt)
    print(f'[RealMLP] fold {k} on {device}: AUC {roc_auc_score(y[va], pv):.5f} ({time.time() - t0:.0f}s)', flush=True)
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

**RealMLP features and launch.** Demidov's feature engineering is built here on train + test, saved together with
the folds and configuration, and the two worker processes are started (one per GPU). Their progress is echoed
here every 30 s; predictions come back as `.npy` files.

In [ ]:
def realmlp_features(tr_df, te_df):
    Xa, Xt = tr_df[BASE].copy(), te_df[BASE].copy()
    cat_cols = Xa.select_dtypes(exclude='number').columns.tolist()
    num_cols = Xa.select_dtypes(include='number').columns.tolist()
    for D in (Xa, Xt):
        D[cat_cols] = D[cat_cols].fillna('missing'); D[num_cols] = D[num_cols].fillna(0.0)
    for c in num_cols:                                          # every numeric value also as a category
        codes_, uniq = Xa[c].factorize()
        Xa[f'{c}_cat_'] = pd.Categorical(codes_)
        Xt[f'{c}_cat_'] = pd.Categorical(Xt[c].map({v: i for i, v in enumerate(uniq)}).fillna(-1).astype('int32'))
    for c in cat_cols + ['Leg room service']:                   # count encodings
        vc = Xa[c].value_counts()
        Xa[f'_{c}_count'] = Xa[c].map(vc).fillna(0).astype('int32'); Xt[f'_{c}_count'] = Xt[c].map(vc).fillna(0).astype('int32')
    combos = [('Class', 'Type of Travel', 'Gender'), ('Class', 'Type of Travel', 'Age'),
              ('Flight Distance', 'Departure/Arrival time convenient')]
    names = []
    for cols in combos:                                         # interaction keys (target-encoded in the worker)
        n = '_'.join(cols) + '_'; names.append(n)
        sa = Xa[cols[0]].astype(str); st = Xt[cols[0]].astype(str)
        for c in cols[1:]:
            sa = sa + '_' + Xa[c].astype(str); st = st + '_' + Xt[c].astype(str)
        codes_, uniq = pd.factorize(sa)
        Xa[n] = pd.Categorical(codes_)
        Xt[n] = pd.Categorical(st.map({v: i for i, v in enumerate(uniq)}).fillna(-1).astype('int32'))
    for c in cat_cols:
        Xa[c] = Xa[c].astype('category'); Xt[c] = Xt[c].astype('category')
    return Xa, Xt, names

RMLP_PARAMS = {
    'random_state': SEED, 'verbosity': 0, 'val_metric_name': '1-auc_ovr',
    'n_ens': 2 if DEBUG else 8, 'n_epochs': 1 if DEBUG else 3, 'batch_size': 256, 'use_early_stopping': False,
    'lr': 0.053, 'wd': 0.015, 'sq_mom': 0.988, 'lr_sched': 'flat_anneal', 'wd_sched': 'cos_log_15',
    'first_layer_lr_factor': 0.25, 'embedding_size': 5, 'max_one_hot_cat_size': 1, 'hidden_sizes': [512, 256, 128],
    'act': 'silu', 'p_drop': 0.05, 'p_drop_sched': 'pow5', 'plr_hidden_1': 16, 'plr_hidden_2': 8,
    'plr_act_name': 'gelu', 'plr_lr_factor': 0.1151, 'plr_sigma': 2.33, 'ls_eps': 0.01, 'ls_eps_sched': 'sqrt_cos',
    'add_front_scale': False, 'bias_init_mode': 'neg-uniform-dynamic-2',
    'tfms': ['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'],
}
Xr, Xr_test, combo_names = realmlp_features(train, test)
pd.to_pickle(dict(X=Xr, X_test=Xr_test, y=y, folds=FOLDS, params=RMLP_PARAMS, seed=SEED, combo_names=combo_names,
                  drop_after_te=['Class_Type of Travel_Age_', 'Flight Distance_Departure/Arrival time convenient_']),
             f'{OUT_DIR}/rmlp_data.pkl')

t0 = time.time()
procs = []
for g in range(N_WORKERS):
    shard = [k for k in range(N_FOLDS) if k % N_WORKERS == g]
    env = dict(os.environ)
    if N_GPUS:
        env['CUDA_VISIBLE_DEVICES'] = str(g)
    log = open(f'{OUT_DIR}/rmlp_worker{g}.log', 'w')
    p = subprocess.Popen([sys.executable, 'realmlp_worker.py', OUT_DIR, ','.join(map(str, shard)),
                          'cuda:0' if N_GPUS else 'cpu'], stdout=log, stderr=subprocess.STDOUT, env=env)
    procs.append((p, log, f'{OUT_DIR}/rmlp_worker{g}.log'))
    print(f'worker {g}: folds {shard} on', f'GPU {g}' if N_GPUS else 'CPU')

seen = set()
while True:
    done = all(p.poll() is not None for p, _, _ in procs)
    for _, _, path in procs:
        for line in open(path, encoding='utf-8', errors='replace'):
            if line.startswith('[RealMLP]') and line not in seen:
                seen.add(line); print(' ', line.strip(), flush=True)
    if done:
        break
    time.sleep(5 if DEBUG else 30)

for p, log, path in procs:
    log.close()
    if p.returncode != 0:
        print(open(path, encoding='utf-8', errors='replace').read()[-4000:])
        raise RuntimeError(f'RealMLP worker failed (exit code {p.returncode}) - log above')

oof = np.zeros(n_train); pred = np.zeros(len(X_test))
for k, (tr, va) in enumerate(FOLDS):
    oof[va] = np.load(f'{OUT_DIR}/rmlp_oof_{k}.npy'); pred += np.load(f'{OUT_DIR}/rmlp_test_{k}.npy') / N_FOLDS
os.remove(f'{OUT_DIR}/rmlp_data.pkl')                     # large intermediate, not needed in the output
results['realmlp'] = dict(oof=oof, pred=pred, score=roc_auc_score(y, oof))
np.save(f'{OUT_DIR}/oof_realmlp.npy', oof); np.save(f'{OUT_DIR}/test_realmlp.npy', pred)
print(f'>>> realmlp: OOF AUC {results["realmlp"]["score"]:.5f}   ({(time.time() - t0) / 60:.1f} min)')

## 8. Ensemble
### 8.1 Model comparison
OOF AUC per model and the rank correlation between models — the lower the correlation, the more a blend can gain.

In [ ]:
names = list(results)
display(pd.Series({n: results[n]['score'] for n in names}, name='OOF AUC').sort_values(ascending=False).round(5).to_frame())
display(pd.DataFrame({n: results[n]['oof'] for n in names}).corr(method='spearman').round(4))

### 8.2 Logistic stacking vs. weighted rank blend
Both are scored **honestly**: the stacker / blend weights are fitted on 4 folds and evaluated on the 5th, using the
same folds as the models. The better of the two (by that honest OOF AUC) produces the submission.

In [ ]:
clip = lambda p: np.clip(p, 1e-6, 1 - 1e-6)
L_oof = np.column_stack([logit(clip(results[n]['oof'])) for n in names])
L_test = np.column_stack([logit(clip(results[n]['pred'])) for n in names])
R_oof = np.column_stack([rankdata(c) / len(c) for c in L_oof.T])
R_test = np.column_stack([rankdata(c) / len(c) for c in L_test.T])

def fit_w(P, t):
    f = lambda w: -roc_auc_score(t, P @ (np.abs(w) / np.abs(w).sum()))
    w = np.abs(minimize(f, np.ones(P.shape[1]) / P.shape[1], method='Nelder-Mead',
                        options={'maxiter': 400, 'xatol': 1e-4}).x)
    return w / w.sum()

stack_oof = np.zeros(n_train); blend_oof = np.zeros(n_train)
for tr, va in FOLDS:
    stack_oof[va] = LogisticRegression(max_iter=2000).fit(L_oof[tr], y[tr]).decision_function(L_oof[va])
    blend_oof[va] = R_oof[va] @ fit_w(R_oof[tr], y[tr])
stack_auc, blend_auc = roc_auc_score(y, stack_oof), roc_auc_score(y, blend_oof)

stacker = LogisticRegression(max_iter=2000).fit(L_oof, y)
w = fit_w(R_oof, y)
best = max(names, key=lambda n: results[n]['score'])
print(f'best single ({best}):   {results[best]["score"]:.5f}')
print(f'rank blend (honest):   {blend_auc:.5f}   weights {dict(zip(names, w.round(3)))}')
print(f'logistic stack (honest): {stack_auc:.5f}   coefs {dict(zip(names, stacker.coef_[0].round(3)))}')

if stack_auc >= blend_auc:
    final_pred, method = stacker.predict_proba(L_test)[:, 1], 'logistic stack'
else:
    final_pred, method = R_test @ w, 'rank blend'
print('Submission uses:', method)

## 9. Submission
Writes `submission.csv` in `sample_submission.csv` order. After *Save Version → Save & Run All*, submit it from the
version's **Output** section or via the competition's *Submit Prediction* page.

In [ ]:
sub = sample[['id']].merge(pd.DataFrame({'id': test['id'].values, TARGET: final_pred}), on='id', how='left')
assert sub[TARGET].notna().all() and len(sub) == len(sample)
sub.to_csv(f'{OUT_DIR}/submission.csv', index=False)
print(f'Saved {OUT_DIR}/submission.csv  shape={sub.shape}')
print(sub[TARGET].describe().round(4))
sub.head()

## Credits
Ideas measured and published by others in this competition, re-implemented here:
* original data — [Airline Passenger Satisfaction](https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction) by teejmahal20;
* `org`, `origm`, `digit`, `freq`, `te` blocks and the ten key pairs — evgendvorkin, [S6E10 LightGBM CV 5 folds 0.96102](https://www.kaggle.com/code/evgendvorkin/s6e10-lightgbm-cv-5-folds-0-96102);
* `Flight Distance` as a route ID and the route profile — goodpjw2008's S6E10 series;
* the block-by-block measurements and the "lookup, not rows" finding — [Route or distance? Both](https://www.kaggle.com/code/busyaprime/s6e10-route-or-distance-both-lb-0-96169);
* RealMLP configuration and features — Vladimir Demidov, [PS|S6|E10: RealMLP · PyTabKit](https://www.kaggle.com/code/yekenot/ps-s6-e10-realmlp-pytabkit).